In [1]:
# Install packages
!pip install -q unsloth datasets transformers accelerate bitsandbytes peft trl

In [2]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Reload the filtered dataset
from datasets import load_dataset
filtered_ds = load_dataset('json', data_files='/content/drive/MyDrive/filtered_ds.jsonl', split='train')
print(f"Reloaded {len(filtered_ds)} examples")

Generating train split: 0 examples [00:00, ? examples/s]

Reloaded 695 examples


In [4]:
!git clone https://github.com/qvd808/dafny-verify-loop.git /content/dafny-verify-loop

# Dafny
!wget -q https://github.com/dafny-lang/dafny/releases/download/v4.4.0/dafny-4.4.0-x64-ubuntu-20.04.zip
!unzip -o dafny-4.4.0-x64-ubuntu-20.04.zip -d /usr/local/dafny
!chmod +x /usr/local/dafny/dafny/dafny
!ln -sf /usr/local/dafny/dafny/dafny /usr/local/bin/dafny

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/DeepSeek-R1-Distill-Llama-8B-bnb-4bit as a legacy tokenizer.


In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Check GPU memory (should be nearly 0 MiB)
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

0 MiB


In [5]:
import os
import torch
from unsloth import FastLanguageModel
from unsloth.trainer import SFTTrainer
from transformers import TrainingArguments

# ---- Memory fragmentation fix ----
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- Settings ----
max_seq_length = 2048

# ---- Load fresh 4‑bit model + tokenizer ----
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = torch.float16,
    load_in_4bit = True,
    device_map = "cuda:0",
)

# ---- Attach LoRA ----
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    max_seq_length = max_seq_length,
)

# ---- Training arguments (3 epochs on filtered 695 examples) ----
training_args = TrainingArguments(
    output_dir = "/content/drive/MyDrive/dafny_finetuned_clean",
    num_train_epochs = 3,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    warmup_steps = 10,
    logging_steps = 5,
    learning_rate = 2e-4,
    fp16 = True,
    bf16 = False,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    save_strategy = "epoch",
    report_to = "none",
)

# ---- Trainer on filtered dataset ----
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = filtered_ds,       # <-- the clean 695-example dataset
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = training_args,
)

# ---- Train ----
trainer.train()

# ---- Save adapter ----
model.save_pretrained("/content/drive/MyDrive/dafny_model_lora_clean")
tokenizer.save_pretrained("/content/drive/MyDrive/dafny_model_lora_clean")
print("✅ Training complete and adapter saved.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/DeepSeek-R1-Distill-Llama-8B-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/695 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 695 | Num Epochs = 3 | Total steps = 261
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.249965
10,2.108512
15,1.443437
20,1.066895
25,0.940058
30,0.815615
35,0.848208
40,0.750429
45,0.780152
50,0.764913


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dafny_finetuned_clean/checkpoint-87/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dafny_finetuned_clean/checkpoint-174/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dafny_finetuned_clean/checkpoint-261/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dafny_model_lora_clean/tokenizer_config.json.


✅ Training complete and adapter saved.


In [7]:
import sys
from pathlib import Path

# Go to the repo root and add it to the Python path
%cd /content/dafny-verify-loop
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Now import
from src import llm

/content/dafny-verify-loop


In [9]:
from src import llm
FastLanguageModel.for_inference(model)

# Re‑define _ANNOTATE_SYSTEM (the same as before)
_ANNOTATE_SYSTEM = """You are a Dafny verification expert. The program below has a complete
code body but is MISSING loop invariants, decreases clauses, and proof assertions.

Your job: add the MINIMAL set of annotations (invariants, decreases, asserts) needed
for Dafny to verify the program.

RULES:
- The existing code body is CORRECT — do NOT change any executable statements.
- Only ADD: invariant clauses, decreases clauses, assert statements, ghost variables.
- Do NOT change method signatures, requires, or ensures clauses.
- Loop invariants must be: true on entry, preserved by the body, strong enough for postcondition.
- decreases must be non-negative integer that strictly decreases.
- Include bounds invariants for all array/sequence accesses.

Output ONLY the complete method/function bodies with annotations added.
Include the full original code plus your annotations. No markdown fences.
"""

def local_chat_completion(system, user, llm_task="formal", temperature=0.0):
    full_prompt = f"{system}\n\n### User:\n{user}\n\n### Assistant:\n"
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=2048).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=1024, do_sample=False, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    for stop_marker in ["### User", "### Human", "### Assistant"]:
        if stop_marker in generated_text:
            generated_text = generated_text.split(stop_marker)[0].strip()
            break
    return generated_text.strip()

llm.chat_completion = local_chat_completion
print("✅ Patched.")

✅ Patched.


In [10]:
from pathlib import Path
from src.pipeline import run_annotate_pipeline

ok, _, _ = run_annotate_pipeline(Path("benchmark/problems/Clover_abs_no_hints.dfy"), max_iters=4, llm_task="formal", verbose=True)
print("Success:", ok)

--- Annotating: Clover_abs_no_hints.dfy ---
Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MES

Success: False


In [ ]:
from datasets import load_dataset

ds = load_dataset("catherinemei/DafnyBench", split="train")

# Your existing system prompt (exact copy)
_ANNOTATE_SYSTEM = """You are a Dafny verification expert. The program below has a complete
code body but is MISSING loop invariants, decreases clauses, and proof assertions.

Your job: add the MINIMAL set of annotations (invariants, decreases, asserts) needed
for Dafny to verify the program.

RULES:
- The existing code body is CORRECT — do NOT change any executable statements.
- Only ADD: invariant clauses, decreases clauses, assert statements, ghost variables.
- Do NOT change method signatures, requires, or ensures clauses.
- Loop invariants must be: true on entry, preserved by the body, strong enough for postcondition.
- decreases must be non-negative integer that strictly decreases.
- Include bounds invariants for all array/sequence accesses.

Output ONLY the complete method/function bodies with annotations added.
Include the full original code plus your annotations. No markdown fences.
"""

def format_annotate_example(example):
    # example['prompt']  = code with annotations removed
    # example['completion'] = code with annotations filled in
    user_message = f"""Add loop invariants, decreases clauses, and proof assertions to make this Dafny program verify.

CURRENT PROGRAM (annotations removed — does NOT verify):
{example['prompt']}

Add the missing annotations. Output the COMPLETE program with annotations.
"""
    # Combine system + user + assistant into a single training text
    full_text = f"{_ANNOTATE_SYSTEM}\n\n### User:\n{user_message}\n\n### Assistant:\n{example['completion']}"
    return {"text": full_text}

formatted_ds = ds.map(format_annotate_example, remove_columns=ds.column_names)

In [ ]:
from datasets import load_dataset

ds = load_dataset("wendy-sun/DafnyBench", split="test")

_ANNOTATE_SYSTEM = ...
def format_annotate_example(example):
    # (same as before)
    user_message = f"""Add loop invariants, decreases clauses, and proof assertions to make this Dafny program verify.

CURRENT PROGRAM (annotations removed — does NOT verify):
{example['hints_removed']}

Add the missing annotations. Output the COMPLETE program with annotations.
"""
    full_text = f"{_ANNOTATE_SYSTEM}\n\n### User:\n{user_message}\n\n### Assistant:\n{example['ground_truth']}"
    return {"text": full_text}

# Tokenize the whole dataset to get lengths
def tokenize_fn(example):
    # Use same tokenizer (already loaded from training) and same max_length=2048 for measurement
    tokens = tokenizer(example["text"], add_special_tokens=True, truncation=False)
    return {"num_tokens": len(tokens["input_ids"])}

ds = ds.map(format_annotate_example, remove_columns=ds.column_names)
ds = ds.map(tokenize_fn)

import numpy as np
lengths = ds["num_tokens"]
print(f"Min: {min(lengths)}, Max: {max(lengths)}, Mean: {np.mean(lengths):.0f}, Median: {np.median(lengths):.0f}")
print(f"Examples <= 2048 tokens: {sum(1 for l in lengths if l <= 2048)} out of {len(lengths)}")

# Keep only those ≤ 2048
filtered_ds = ds.filter(lambda x: x["num_tokens"] <= 2048)
print(f"Filtered dataset size: {len(filtered_ds)}")

In [37]:
# Save the filtered dataset (695 examples) to Drive so we don't lose it after restart
filtered_ds.to_json("/content/drive/MyDrive/filtered_ds.jsonl")
print("✅ filtered_ds saved to Drive.")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✅ filtered_ds saved to Drive.
